In [1]:
## Basic stuff
%load_ext autoreload
%autoreload

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))
#IPython.Cell.options_default.cm_config.lineNumbers = true;


###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 

from time import sleep
from glob import glob
from ioUtils import saveFile, getFile
from fsUtils import isFile
from timeUtils import clock, elapsed, timestat
from fileIO import fileIO
from masterDBGate import masterDBGate
from masterDBData import masterDBData
from masterManualEntries import masterManualEntries
from masterManualEntriesUtils import masterManualEntriesUtils
from masterUtils import masterUtils
from matchArtistToDB import masterMatch
from pandas import Series, DataFrame, isna, notna


# Global

In [3]:
io      = fileIO()
mdbGate = masterDBGate()
mu      = masterUtils()
disc    = mu.getDisc("MusicBrainz")

# Collect External Data

In [ ]:
externalData = {"Discogs": {}, "LastFM": {}, "RateYourMusic": {}, "Deezer": {}, "AllMusic": {}, "Genius": {}}

In [ ]:
externalLinks = {}
for modVal in range(100):
    modValData = disc.getDBModValData(modVal)
    for artistID,artistIDData in modValData.iteritems():
        externalIDData = artistIDData.profile.external        
        if isinstance(externalIDData, dict):
            for dbName,dbNameData in externalIDData.items():
                externalLinksData = [link.href for link in dbNameData] if isinstance(dbNameData, list) else []
                if len(externalLinksData) > 0:
                    if externalLinks.get(artistID) is None:
                        externalLinks[artistID] = {}
                    externalLinks[artistID][dbName] = externalLinksData
    if (modVal+1)%2 == 0:
        print("{0: <6}{1}".format(modVal+1,len(externalLinks)))

# Master External DF

In [ ]:
ts = timestat("Creating Master External DataFrame")
externalLinksDF = Series(externalLinks).apply(Series)
ts.stop()

In [ ]:
ts = timestat("Creating External <=> Artist DataFrame")
artistIDToNameData = disc.getArtistIDToNameData()
externalArtistData = artistIDToNameData.loc[externalLinksDF.index]
externalArtistData.name = "ArtistName"
externalArtistData = DataFrame(externalArtistData)
externalArtistData = externalArtistData.join(externalLinksDF)
ts.stop()
externalArtistData.head()

In [ ]:
externalArtistData.count().sort_values()

In [ ]:
io.save(idata=externalArtistData, ifile="MusicBrainzExternalLinks.p")

# Extract Artist IDs

In [4]:
externalArtistData = io.get("MusicBrainzExternalLinks.p")

In [5]:
from pandas import isna, notna

def fixEntry(x):
    if isinstance(x, list):
        if len(x) == 1:
            return x[0]
        else:
            return x
    elif isna(x):
        return None
    else:
        raise ValueError("Unsure what to do with {0}".format(x))

externalArtistData = io.get("MusicBrainzExternalLinks.p")
for col,colData in externalArtistData.drop(['ArtistName'], axis=1).iteritems():
    externalArtistData[col] = colData.apply(fixEntry)

## AllMusic

In [43]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()
mmeDF = meu.getDF()

========================= masterManualEntries(install=False) =========================
Current Time Is 2022-01-26 20:31:00
Process [Getting Manual Entries Data From Main Pickle File] Took 1.7 Seconds
Current Time Is 2022-01-26 20:31:00
========================= MasterDB Data(dtype=Search, dbs=None) =========================


In [44]:
mmeDF
#meu.saveNewDF(mmeDF.drop(["Deezer", "LastFM"], axis=1))

,Rank,Albums,Counts,ArtistName,Discogs,AllMusic,MusicBrainz,RateYourMusic,DeezerAPI,LastFMAPI,Genius,AlbumOfTheYear,KWorbiTunes,KWorbSpotify,SpotifyCharts,Spotify,Soundcloud,Tidal
ggggggggXXX0023714XXX1,1.0,3536.0,13.0,Gucci Mane,419677,0000545523,161172248151233562465037800043206339445,80296,7972,38195921803,93510271,1648,638441780223,502009748861,10992308341,13y7CgLHjMVRMDqxdx0Xdo,None,55577
ddddddddXXX0014134XXX1,2.0,3539.0,13.0,David Guetta,36527,0000213561,173367809260979027817314779471947924979,20772,542,18135150764,40145811,1145,333969078948,550854821902,61570109868,1Cs0zKBU1kc0i8ypK3B9ai,davidguetta,27343
aaaaaaaaXXX0042707XXX1,3.0,8118.0,13.0,Armin Van Buuren,9070,0000502989,285287515645240787159302376016051220362,54213,10620,13508042519,79591979,4022,800051365617,139198636210,35370791250,0SfsnGyD8FpIN4U4WCkBZ5,arminvanbuuren,3521263
jjjjjjjjXXX0050344XXX1,4.0,3033.0,13.0,Justin Bieber,1602787,0002165952,265190129791694491272726226394162286533,473105,288166,43125808998,46023405,2438,174285989430,316532562867,12040482714,1uNFoZAHBGtllmzznpCI3s,justinbieber,3639248
ccccccccXXX0030029XXX1,5.0,3321.0,13.0,Coldplay,29735,0000775877,329477306239508679718125499404026181388,138,892,44983268338,14724250,62,936137447686,917391174586,99841519945,4gzpq5DPGxSnKTe4SA8HAU,coldplay,8812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
llllllllXXX0020824XXX1,828025.0,0.0,2.0,Lizards,556149,None,113091944725517147387624857428517939595,None,None,None,None,None,None,None,None,None,None,None
llllllllXXX0020823XXX1,828025.0,0.0,2.0,Lizardpotion,9488536,None,300313129405555288322707293352745005234,None,None,None,None,None,None,None,None,None,None,None
llllllllXXX0020841XXX1,828025.0,0.0,2.0,Lizoel Costa,3818095,None,96867576028477941114780978704931106667,None,None,None,None,None,None,None,None,None,None,None
00000033XXX0000001XXX1,828025.0,0.0,2.0,! Obtain?,2738698,None,197709043848520256131813846007099605285,None,None,None,None,None,None,None,None,None,None,None


In [12]:
dbMatches = {db: externalArtistData[db][externalArtistData[db].notna()] for db in ["AllMusic", "Spotify", "Deezer", "Discogs", "Genius", "LastFM", "Soundcloud", "Tidal"]}

In [17]:
knownMusicBrainzIDs = mmeDF["MusicBrainz"][mmeDF["MusicBrainz"].notna()]
possibleNewArtists  = Series()

for db,dbMatch in dbMatches.items():
    print("")
    print("="*35,db,"="*35)
    
    print("There are {0} Known MusicBrainz IDs".format(knownMusicBrainzIDs.shape[0]))
    print("There are {0} MusicBrainz <=> {1} Matches".format(len(dbMatch), db))
    knownMusicMatch = dbMatch[dbMatch.index.isin(knownMusicBrainzIDs)]
    print("There are {0} Known MusicBrainz IDs out of the {1} MusicBrainz <=> {2} Matches".format(knownMusicMatch.shape[0], dbMatch.shape[0], db))
    newknownMusicMatch = dbMatch[~dbMatch.index.isin(knownMusicBrainzIDs)]
    print("There are {0} New MusicBrainz IDs out of the {1} <=> {2} Matches".format(newknownMusicMatch.shape[0], dbMatch.shape[0], db))
    
    possibleNewArtists = possibleNewArtists.append(newknownMusicMatch)


=================================== AllMusic ===================================
There are 810286 Known MusicBrainz IDs
There are 112529 MusicBrainz <=> AllMusic Matches
There are 111567 Known MusicBrainz IDs out of the 112529 MusicBrainz <=> AllMusic Matches
There are 962 New MusicBrainz IDs out of the 112529 <=> AllMusic Matches

=================================== Spotify ===================================
There are 810286 Known MusicBrainz IDs
There are 101927 MusicBrainz <=> Spotify Matches
There are 70368 Known MusicBrainz IDs out of the 101927 MusicBrainz <=> Spotify Matches
There are 31559 New MusicBrainz IDs out of the 101927 <=> Spotify Matches

=================================== Deezer ===================================
There are 810286 Known MusicBrainz IDs
There are 44312 MusicBrainz <=> Deezer Matches
There are 43943 Known MusicBrainz IDs out of the 44312 MusicBrainz <=> Deezer Matches
There are 369 New MusicBrainz IDs out of the 44312 <=> Deezer Matches

============

In [45]:
newArtists = externalArtistData.loc[possibleNewArtists.index.drop_duplicates(), ["Discogs", "AllMusic", "Deezer", "LastFM", "Spotify", "Genius", "Soundcloud", "Tidal", "RateYourMusic"]].copy(deep=True)

In [24]:
def getIDs(x, db):
    if isinstance(x, list):
        return [mu.getid(db, val) for val in x]
    else:
        return mu.getid(db, x)

newArtistIDs = {db: dbData.apply(getIDs, db=db) for db,dbData in newArtists.iteritems()}

In [33]:
artistNames = mu.getArtistNameData("MusicBrainz", "DB")

In [36]:
df = DataFrame(newArtistIDs)
df["Counts"] = df.apply(lambda x: x.count(), axis=1)
df.index.name = "MusicBrainz"
df.reset_index(inplace=True)
df["ArtistName"] = df["MusicBrainz"].apply(lambda x: artistNames.get(x))
df = df.sort_values(by="Counts", ascending=False)

In [38]:
io.save(idata=df, ifile="newArtistsFromMusicBrainzExternalData.p")

In [28]:
externalArtistData["Soundcloud"][externalArtistData["Soundcloud"].notna()]

108541848016828757278131944962756872900             https://soundcloud.com/prince3eg
239845760866843263930154809723580934900           https://soundcloud.com/richardmarx
172552485256597266680385033568580864600         https://soundcloud.com/the-romantics
271799920922248872742414459945755823000    https://soundcloud.com/chris-rea-official
273044408241922325766593344495591987500     https://soundcloud.com/thedoobiebrothers
                                                             ...                    
229462969129925582506409148259487537699      https://soundcloud.com/littlehumanmusic
242875680075815623712729516917947743499              https://soundcloud.com/lil-lolu
291141302739599796156038890174499035399            https://soundcloud.com/kaishandao
164920222063856373167381158843806118299              https://soundcloud.com/milyma-1
72688589373675558605493430474252264399            https://soundcloud.com/muta-musica
Name: Soundcloud, Length: 101772, dtype: object

## Master Key

In [71]:
from uuid import uuid4

class masterKey(str):
    def __init__(self):
        self.key = "".join(str(uuid4()).split('-')[:3])
        
    def get(self):
        return self.key
    
mk = masterKey()
mk

''

## Artist Counts

In [67]:
artistCounts = mmeDF["ArtistName"].str.upper().value_counts()
mmeDF["ArtistName"].str.upper().apply(artistCounts.get)

## Artist Type, Genre, Tags

### Spotify

In [78]:
disc = mu.getDisc("Spotify")

def getSpotifyType(x):
    retval = x.profile.general.get('Type') if isinstance(x.profile.general, dict) else None
    return retval

def getSpotifyGenres(x):
    retval = x.profile.genres if x.profile.genres is not None else None
    return retval

from pandas import concat
spotifyTypes  = concat([disc.getDBModValData(modVal).apply(getSpotifyType) for modVal in range(100)])
spotifyGenres = concat([disc.getDBModValData(modVal).apply(getSpotifyGenres) for modVal in range(100)])

#for idx,item in modValData.iteritems():
#    print(item.profile.get())
#    break

In [80]:
spotifyTypes.value_counts()

artist    533366
dtype: int64

In [ ]:

spotifyIDs = externalArtistData["Spotify"].apply(getIDs, db="Spotify")
spotifyIDs[spotifyIDs == 'None'] = None

soundcloudIDs = externalArtistData["Soundcloud"].apply(getIDs, db="Soundcloud")
soundcloudIDs[soundcloudIDs == 'None'] = None

tidalIDs = externalArtistData["Tidal"].apply(getIDs, db="Tidal")
tidalIDs[tidalIDs == 'None'] = None

In [ ]:
meu = masterManualEntriesUtils()
mmeDF = meu.getDF().copy(deep=True)

In [ ]:
#from pandas import merge
#mmeDFMerge = merge(mmeDF, spotifyIDDF[spotifyIDDF["Spotify"].notna()], on=["MusicBrainz"]) #, how='left')
#mmeDFMerge.head()
mmeDF["Spotify"]    = mmeDF["MusicBrainz"].apply(spotifyIDs.get)
mmeDF["Soundcloud"] = mmeDF["MusicBrainz"].apply(soundcloudIDs.get)
mmeDF["Tidal"]      = mmeDF["MusicBrainz"].apply(tidalIDs.get)

In [ ]:
mmeDF.head()

In [ ]:
DataFrame(mmeDF["MusicBrainz"]).jo

In [ ]:
aid = artistIDGenius()
aid.getArtistID("https://genius.com/artists/Prince")

In [ ]:
aid = artistIDLastFM()
aid.getArtistID("https://www.last.fm/music/Richard+Marx")

In [ ]:
aid = artistIDSpotify()
aid.getArtistID("https://open.spotify.com/artist/5a2EaR3hamoenG9rDuVn8j")

In [ ]:
'5a2EaR3hamoenG9rDuVn8j'
'5a2EaR3hamoenG9rDuVn8j'

In [ ]:
externalArtistData["YouTube"][externalArtistData["YouTube"].notna()].apply(lambda x: x.split("/")[-2:] if isinstance(x,str) else []).apply(lambda x: x[0] if len(x) > 0 else "").value_counts()

In [ ]:
disc = mdbGate.getDBDisc("AlbumOfTheYear")
disc.getArtistIDToRefData().head().to_list()

In [ ]:
externalArtistData.head().T

In [ ]:
def getIDs(x, db):
    if isinstance(x,str):
        retval = utils[db].getArtistID(x)
        retval = retval if (isinstance(retval, str) and retval.isdigit()) else x
        return retval
    elif isinstance(x,list):
        retval = [getIDs(val, db) for val in x]
        return retval
    elif x is None:
        return None
    else:
        raise ValueError("Unsure what to do with {0}".format(x))
        
for col,colData in externalArtistData.drop(['ArtistName', 'RateYourMusic', 'Genius'], axis=1).iteritems():
    print(col)
    externalArtistData[col] = externalArtistData[col].apply(lambda x: getIDs(x, db=col))

In [ ]:
mme   = masterManualEntries()
mmeDF = mme.getData()

In [ ]:
knownMusicBrainzIDs            = mmeDF["MusicBrainz"][mmeDF["MusicBrainz"].notna()]
print("There are {0} Previously Matched MusicBrainz IDs".format(knownMusicBrainzIDs.shape[0]))
print("There are {0} External MusicBrainz IDs <=> DB IDs".format(externalArtistData.shape[0]))
knownExternalMusicBrainzData   = externalArtistData[externalArtistData.index.isin(knownMusicBrainzIDs)]
print("There are {0} Previously Matched External MusicBrainz IDs <=> DB IDs".format(knownExternalMusicBrainzData.shape[0]))
unknownExternalMusicBrainzData = externalArtistData[~externalArtistData.index.isin(knownMusicBrainzIDs)]
print("There are {0} Unmatched External MusicBrainz IDs <=> DB IDs".format(unknownExternalMusicBrainzData.shape[0]))

# Check and Append Known External MusicBrainz Data

## Test For Correct Assignments

In [ ]:
dbsToMatch = list(unknownExternalMusicBrainzData.columns.drop(["ArtistName", "RateYourMusic", "Genius",  "LastFM"]))

In [ ]:
masterDF = mmeDF[mmeDF["MusicBrainz"].isin(knownExternalMusicBrainzData.index)][["MusicBrainz"]+dbsToMatch].copy(deep=True)

In [ ]:
knownExternalMusicBrainzData = knownExternalMusicBrainzData.reset_index().rename(columns={"index": "MusicBrainz"})

In [ ]:
from pandas import merge
masterExternalDF = merge(masterDF, knownExternalMusicBrainzData[["MusicBrainz"]+dbsToMatch], on="MusicBrainz", suffixes=("_Master", "_External"))
masterExternalDF = merge(masterExternalDF, mmeDF[["ArtistName","MusicBrainz"]], on="MusicBrainz")



from pandas import isna, notna
def fixEntry(x):
    if isinstance(x, str) and x.isdigit():
        return x
    elif isinstance(x, str) and not x.isdigit():
        return x
    elif isinstance(x, float):
        if isna(x):
            return None
        else:
            return str(x)
    elif isinstance(x, list):
        return x
    elif x is None:
        return None
    else:
        raise ValueError("Unsure how to parse {0}".format(x))
        
for col in dbsToMatch:
    masterExternalDF["{0}_Master".format(col)] = masterExternalDF["{0}_Master".format(col)].apply(fixEntry)
    masterExternalDF["{0}_External".format(col)] = masterExternalDF["{0}_External".format(col)].apply(fixEntry)

In [ ]:
masterExternalDF.head()

In [ ]:
def compareDBIDs(row):
    retval = {}
    for db in dbsToMatch:
        masterID   = row["{0}_Master".format(db)]
        externalID = row["{0}_External".format(db)]
        if isinstance(masterID,str):
            if isinstance(externalID,str):
                if externalID.isdigit():
                    retval[db] = masterID == externalID
                else:
                    retval[db] = "URL"
            elif isinstance(externalID,list):
                retval[db] = masterID in externalID
            elif externalID is None:
                retval[db] = None
            else:
                raise ValueError("Unsure how to handle {0}/{1}, ID={2}".format(db,externalID,row.name))
        elif masterID is None:
            if isinstance(externalID,str):
                if externalID.isdigit():
                    retval[db] = externalID
                else:
                    retval[db] = "URL"
            elif isinstance(externalID,list):
                retval[db] = externalID
            elif externalID is None:
                retval[db] = None
            else:
                raise ValueError("Unsure how to handle {0}/{1}, ID={2}".format(db,externalID,row.name))
        else:
            raise ValueError("Unsure how to handle {0}/{1}/{2}, ID={3}".format(db,type(masterID),externalID,row.name))
            
    return Series(retval)
            
ts = timestat("Comparing Master and External Results")
masterExternalResults = masterExternalDF.apply(compareDBIDs, axis=1)
ts.stop()

## New DB IDs

In [ ]:
masterExternalDF.loc[newDiscogs.index, "MusicBrainz"]
#merge(masterExternalDF.loc[newDiscogs.index, "MusicBrainz"].reset_index(), mmeDF.reset_index()

In [ ]:
mmeDF[mmeDF["MusicBrainz"].isin(masterExternalDF.loc[newDiscogs.index, "MusicBrainz"])][["MusicBrainz"]].to_dict()

In [ ]:
#newDiscogs = masterExternalResults["Discogs"][masterExternalResults["Discogs"].apply(lambda x: x.isdigit() if isinstance(x, str) else False)]
newDeezer = masterExternalResults["Deezer"][masterExternalResults["Deezer"].apply(lambda x: x.isdigit() if isinstance(x, str) else False)]
newDeezer

In [ ]:
meu = masterManualEntriesUtils()
for idx,row in merge(DataFrame(newDeezer).join(masterExternalDF["MusicBrainz"]), mmeDF.reset_index()[["index", "MusicBrainz"]], on="MusicBrainz").iterrows():
    meu.setdbid(row["index"], "Deezer", row["Deezer"])
    #print("meu.setamid(\'{0}\', \'{1}\')".format(row["index"], row["AllMusic"]))
meu.saveDF()

## To Understand Different IDs

In [ ]:
refData = disc.getArtistIDToRefData()
for idx,row in masterExternalResults[masterExternalResults.eq(False).any(1)].iterrows():
    mbID = masterExternalDF.loc[idx,"MusicBrainz"]
    masterIDX = mmeDF[mmeDF["MusicBrainz"] == mbID].index
    masterIDX = list(masterIDX)[0] if len(masterIDX) == 1 else None
    if masterIDX is None:
        print("Somehow masterIDX is None!")
        print(idx)
        print(row)
        break
        
    for db,dbval in row.iteritems():
        if dbval == False:
            url = refData.get(mbID)
            print("")
            print("="*15,mmeDF.loc[masterIDX,"ArtistName"],"="*15)
            print(url)
            print("meu.setdbid(\'{0}\', \'{1}\', \'{2}\')".format(masterIDX, db, masterExternalDF.loc[idx,"{0}_Master".format(db)]))
            print("meu.setdbid(\'{0}\', \'{1}\', \'{2}\')".format(masterIDX, db, masterExternalDF.loc[idx,"{0}_External".format(db)]))

# Check and Append Unknown External MusicBrainz Data

## Test For Known DBIDs Missing MusicBrainz ID

In [ ]:
dbsToMatch = list(unknownExternalMusicBrainzData.columns.drop(["ArtistName", "RateYourMusic", "Genius"]))

In [ ]:
overlappingMBIDs = []
print("There are {0} External Entries Without A Matching MusicBrainz Match".format(unknownExternalMusicBrainzData.shape[0]))
for col in dbsToMatch: 
    print("="*100)
    knownDBIDs     = mmeDF[col][mmeDF[col].notna()]
    print("There are {0} Matched {1} DB IDs".format(knownDBIDs.shape[0], col))
    externalDBIDs  = unknownExternalMusicBrainzData[col][unknownExternalMusicBrainzData[col].notna()]
    print("There are {0} External+Unknown MusicBrainz {1} DB IDs".format(externalDBIDs.shape[0], col))
    
    overlappingDBIDs = externalDBIDs[externalDBIDs.isin(knownDBIDs)]
    print("There are {0} Overlapping Matched {1} DB IDs + External Unknown MusicBrainz IDs".format(overlappingDBIDs.shape[0], col))
    
    if overlappingDBIDs.shape[0] > 0:
        overlappingMBIDs.append(overlappingDBIDs)

In [ ]:
from pandas import concat
#unknownExternalMusicBrainzData
somewhatKnownIDs = concat(overlappingMBIDs).index.drop_duplicates()
mixUnknownExternalMusicBrainzData = unknownExternalMusicBrainzData.loc[somewhatKnownIDs]
newUnknownExternalMusicBrainzData = unknownExternalMusicBrainzData[~unknownExternalMusicBrainzData.index.isin(somewhatKnownIDs)]

In [ ]:
io.save(idata=mixUnknownExternalMusicBrainzData, ifile="MusicBrainzMixArtists.p")
io.save(idata=newUnknownExternalMusicBrainzData, ifile="MusicBrainzNewArtists.p")

In [ ]:
newUnknownExternalMusicBrainzData = io.get("MusicBrainzNewArtists.p")

In [ ]:
from uuid import uuid4
meu = masterManualEntriesUtils()
newRows = []
for mbid,row in newUnknownExternalMusicBrainzData.iterrows():
    rowData=row[row.notna()].drop(["ArtistName"]).to_dict()
    rowData["MusicBrainz"] = mbid
    
    newRow = {"ArtistName": row["ArtistName"]}
    newRow.update(rowData)
    newRow = Series(newRow)
    newRow.name = str(uuid4())
    newRows.append(newRow)
    
    #meu.newArtist(str(row["ArtistName"]), **rowData)
#meu.saveDF()

In [ ]:
mmeDF = meu.getDF()

In [ ]:
mmeDF

In [ ]:
mmeDF = mmeDF.append(newRows)

In [ ]:
mme.saveData(mmeDF)

In [ ]:
mme.reIndexData()

In [ ]:
mmeDF = mme.getData()

In [ ]:
idxs="""2168e9d0-ce22-4a27-9d9f-ff30b8e8fec7            NaN         NaN              
f806a5c4-fc82-4de1-b9fa-613ea1a3786f            NaN         NaN              
ce4b89fc-b151-44c7-9414-31f8f13ed67c            NaN  0000977410              
80f7a571-ee05-49a7-895e-8a1895c39ed1            NaN         NaN              
668993fc-f572-4181-a33d-a93fad72ecab            NaN         NaN              
da5ba952-c806-4aef-9ed5-1139f478e57a            NaN         NaN              
450541aa-ebde-4c41-9cde-6cd6cad15e65            NaN         NaN              
5044ba7f-ff36-4c49-b02e-0c369a44ace0            NaN         NaN              
4e2d0174-a160-4891-b8fd-0875febd33f9            NaN         NaN              
791e8af2-9383-4856-8f4d-dc35ea9d7498            NaN         NaN              
06b721b9-b580-4a2f-a1a0-94c1d707a732            NaN         NaN              
6e71f7de-30ab-40dc-9dd0-736889eeefea            NaN         NaN              
a060f8a8-f1b5-42f5-b3cf-e0922e50d4b4            NaN         NaN              
bf1f8ee5-ef8a-40fa-9bbe-31d78cb6b435            NaN         NaN              
2f1e98de-0caf-4f30-929d-90db6ba68a13            NaN         NaN""".split("\n")
idxs = [x.split()[0] for x in idxs]
idxs

In [ ]:
mmeDF.drop(idxs, axis=0, inplace=True)

In [ ]:
mme.saveData(mmeDF)

In [ ]:
overlappingDBIDs

In [ ]:
unknownExternalMusicBrainzData[unknownExternalMusicBrainzData.index.isin()]

In [ ]:
#externalArtistData.loc[overlappingDBIDs.index]
externalArtistData.loc['15998165970339229035232016575136129888']

In [ ]:
mmeDF[mmeDF["Discogs"].isin(overlappingDBIDs)]

In [ ]:
unknownExternalMusicBrainzData.head()

In [ ]:
mmeDF[mmeDF[col] == '1424']

In [ ]:
mmeDF[mmeDF[col].isin(overlappingIDs)]

In [ ]:

for col,colData in unknownExternalMusicBrainzData.drop(['ArtistName', 'RateYourMusic'], axis=1).iteritems():
    knownDBIDs     = mmeDF[col][mmeDF[col].notna()]
    externalDBIDs  = colData[colData.notna()]
    
    overlappingIDs = externalDBIDs[externalDBIDs.isin(knownDBIDs)]
    for mbID,dbID in externalDBIDs.iteritems():
        ### Test DBID <=> Other DBIDs
        for col2,col2Data in unknownExternalMusicBrainzData.drop([col, 'ArtistName', 'RateYourMusic'], axis=1).iteritems():
            
unknownExternalMusicBrainzData["Discogs"].isin([mmeDF["Discogs"]])

In [ ]:
for col,colData in externalArtistData.drop(['ArtistName', 'RateYourMusic'], axis=1).iteritems():
    externalArtistData[col] = externalArtistData[col].apply(lambda x: getIDs(x, db=col))

In [ ]:
for mbID,row in unknownExternalMusicBrainzData.iteritems()